In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

# Data Understanding

### Column Dictionary

| # | Column | Description | Data Type | Values / Range |
|---|---|---|---|---|
| 1 | `transaction_id` | Unique identifier of the transaction | Text (ID) | e.g. `TXN-20230000000` |
| 2 | `account_id` | Identifier of the bank account that made the transaction | Text (ID) | e.g. `ACC-229636759` |
| 3 | `customer_id` | Identifier of the customer who owns the account | Text (ID) | e.g. `CUST-616210` |
| 4 | `customer_name` | Full name of the customer (not unique — names repeat across customers) | Text | 400 distinct names |
| 5 | `account_type` | Type of bank account | Categorical | Checking, Savings, Credit Card, Loan, Mortgage, Investment |
| 6 | `account_status` | Current status of the account | Categorical | Active, Dormant (inactive), Frozen (blocked), Closed |
| 7 | `transaction_type` | Kind of financial operation performed | Categorical | Deposit, Withdrawal, Transfer, Payment, Refund, Fee, Interest |
| 8 | `transaction_date` | Date the transaction took place | Date (stored as text) | 2022-01-01 → 2025-12-31 |
| 9 | `amount` | Value of the transaction (positive or negative) | Numerical (float) | -4,737,986.89 → 4,091,721.74 |
| 10 | `currency` | Currency of the transaction amount | Categorical | EUR, USD, GBP, CHF, PLN |
| 11 | `balance_after` | Account balance after the transaction | Numerical (float) | 107.18 → 499,987.45 |
| 12 | `transaction_channel` | Channel through which the transaction was made | Categorical | Online, Mobile App, ATM, Branch, Phone, SEPA, Direct Debit |
| 13 | `branch_city` | German city of the bank branch handling the account | Categorical | 15 cities (Berlin, Munich, Hamburg, Frankfurt, …) |
| 14 | `merchant_name` | Merchant or beneficiary of the transaction | Categorical | REWE, Lidl, Aldi, EDEKA, Shell, Amazon, Netflix, MediaMarkt, Deutsche Bahn |
| 15 | `payment_method` | Method used to pay | Categorical | Debit Card, Credit Card, Bank Transfer, SEPA, Cash |
| 16 | `is_international` | Whether the transaction crossed borders | Boolean | True / False |
| 17 | `fee_amount` | Fee charged by the bank on the transaction | Numerical (float) | 0 → 25 |
| 18 | `risk_score` | Automated risk rating of the transaction (higher = riskier) | Numerical (float) | 0 → 100 |
| 19 | `fraud_flag` | Whether the transaction was flagged as fraudulent | Boolean | True / False |
| 20 | `aml_check_status` | Result of the Anti-Money Laundering (AML) check on the transaction | Categorical | Passed, Failed, Pending, Manual Review, Skipped |
| 21 | `kyc_status` | Status of the Know Your Customer (KYC) identity verification | Categorical | Verified, Pending, Expired, Not Required |
| 22 | `credit_score` | Customer's creditworthiness score (higher = more creditworthy) | Numerical (float) | 300 → 850 |
| 23 | `customer_segment` | Customer category used by the bank | Categorical | Retail, Premium, Private Banking, Business, Corporate |
| 24 | `ip_address` | IP address the transaction was made from | Text | e.g. `206.241.176.171` |
| 25 | `created_at` | Timestamp when the record was created in the bank's system | Datetime (stored as text) | 2023-01-01 → 2023-12-28 |

# Data Exploration

## 1. Load Dataset

In [2]:
df=pd.read_csv('finance_banking_transactions_2022_2025.csv')

## 2. Check Data Types

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 52624 entries, 0 to 52623
Data columns (total 25 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   transaction_id       52624 non-null  str    
 1   account_id           52624 non-null  str    
 2   customer_id          52624 non-null  str    
 3   customer_name        52624 non-null  str    
 4   account_type         52624 non-null  str    
 5   account_status       52624 non-null  str    
 6   transaction_type     52624 non-null  str    
 7   transaction_date     52624 non-null  str    
 8   amount               52624 non-null  float64
 9   currency             52624 non-null  str    
 10  balance_after        52624 non-null  float64
 11  transaction_channel  52624 non-null  str    
 12  branch_city          52624 non-null  str    
 13  merchant_name        45378 non-null  str    
 14  payment_method       52624 non-null  str    
 15  is_international     52624 non-null  bool   
 1

## 3. Check Duplicates

In [4]:
df.duplicated()

0        False
1        False
2        False
3        False
4        False
         ...  
52619     True
52620     True
52621     True
52622     True
52623     True
Length: 52624, dtype: bool

In [5]:
df[df.duplicated()]

,transaction_id,account_id,customer_id,customer_name,account_type,account_status,transaction_type,transaction_date,amount,currency,...,is_international,fee_amount,risk_score,fraud_flag,aml_check_status,kyc_status,credit_score,customer_segment,ip_address,created_at
52000,TXN-20230004774,ACC-715158530,CUST-880085,Felix Schneider,Savings,Dormant,Withdrawal,2023-09-30,-1987.42,GBP,...,False,17.36,15.57,False,Pending,Not Required,535.0,Retail,246.125.201.129,2023-03-24 15:13:00
52001,TXN-20230009906,ACC-478409872,CUST-664476,Thomas Schmidt,Checking,ACTIVE,Fee,2024-06-05,11242.83,PLN,...,True,5.00,91.09,True,Skipped,Not Required,NaN,Retail,155.59.252.225,2023-02-26 02:57:00
52002,TXN-20230017925,ACC-199859104,CUST-273981,Christina Schneider,Loan,Frozen,Refund,2022-11-12,49866.17,EUR,...,True,13.54,65.79,True,passed,Verified,442.0,Private Banking,168.81.250.109,2023-11-03 03:50:00
52003,TXN-20230020898,ACC-345760348,CUST-280438,Sabine Schmidt,Credit Card,Dormant,Transfer,2023-12-30,31182.45,PLN,...,True,0.00,64.82,False,Failed,Pending,378.0,Business,194.12.60.245,2023-08-05 03:41:00
52004,TXN-20230041075,ACC-224875363,CUST-919162,Sophie Sowah,Checking,Frozen,Deposit,01 May 2023,-23352.41,USD,...,True,0.00,1.27,True,Manual Review,verified,434.0,Private Banking,169.188.145.22,2023-11-14 09:25:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52619,TXN-20230014029,ACC-653025656,CUST-334886,Christina Sowah,Credit Card,Frozen,Payment,02-10-2022,5753.41,EUR,...,True,0.00,87.58,False,passed,Verified,NaN,Private Banking,9.191.163.119,2023-05-25 20:00:00
52620,TXN-20230039166,ACC-755323946,CUST-152217,Lukas Bauer,Mortgage,ACTIVE,Payment,2023-03-03,-36721.89,EUR,...,True,0.00,49.35,False,Skipped,Verified,546.0,Business,21.201.94.210,2023-07-17 05:00:00
52621,TXN-20230044663,ACC-970876882,CUST-187736,Paul Becker,Investment,Dormant,Payment,30/09/2023,49127.14,EUR,...,True,22.91,17.89,True,Pending,Verified,319.0,Premium,222.99.169.230,2023-03-19 20:55:00
52622,TXN-20230003784,ACC-712988228,CUST-657814,Paul Boateng,Credit Card,Active,Deposit,10 Apr 2023,38328.82,USD,...,True,0.00,39.75,False,Passed,Pending,666.0,Private Banking,197.215.46.180,2023-12-25 18:45:00


In [6]:
df.drop_duplicates()

,transaction_id,account_id,customer_id,customer_name,account_type,account_status,transaction_type,transaction_date,amount,currency,...,is_international,fee_amount,risk_score,fraud_flag,aml_check_status,kyc_status,credit_score,customer_segment,ip_address,created_at
0,TXN-20230000000,ACC-229636759,CUST-616210,Sabine Koch,Checking,Frozen,Transfer,2025-06-14,17101.85,PLN,...,False,0.00,6.28,False,Failed,Pending,592.0,Premium,206.241.176.171,2023-01-13 12:32:00
1,TXN-20230000001,ACC-856806426,CUST-754810,Daniel Boateng,Loan,Frozen,Refund,2022-08-18,45026.38,USD,...,True,0.00,60.60,False,Passed,Expired,686.0,Private Banking,221.29.175.160,2023-02-26 08:58:00
2,TXN-20230000002,ACC-443163624,CUST-644966,Lukas Koch,Credit Card,Closed,Fee,2023-05-18,10658.79,EUR,...,True,0.00,97.67,False,Pending,Expired,763.0,Corporate,199.1.62.209,2023-07-10 15:28:00
3,TXN-20230000003,ACC-949045770,CUST-365821,Mia Weber,Credit Card,active,Interest,2023-09-22,-44113.52,PLN,...,True,13.33,71.52,True,Pending,Pending,388.0,Business,1.156.52.54,2023-02-25 07:14:00
4,TXN-20230000004,ACC-696055802,CUST-161490,Hanna Muller,Mortgage,Closed,Transfer,2022-05-29,-38955.13,CHF,...,True,7.44,51.16,True,Passed,verified,NaN,Corporate,69.62.158.212,2023-03-05 05:06:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51995,TXN-20230051995,ACC-829518324,CUST-440936,Michael Schneider,Mortgage,Frozen,Deposit,2025-02-18,-10566.44,GBP,...,True,0.00,91.69,True,Manual Review,Expired,NaN,Private Banking,162.132.94.134,2023-02-08 00:25:00
51996,TXN-20230051996,ACC-681588259,CUST-266725,Emma Bauer,Credit Card,Frozen,Interest,2022-03-07,10562.28,CHF,...,False,0.00,54.04,True,passed,verified,NaN,Premium,153.223.61.55,2023-01-23 13:14:00
51997,TXN-20230051997,ACC-866271890,CUST-201440,Thomas Sowah,Mortgage,Closed,Deposit,2024-03-09,46737.13,PLN,...,True,0.00,40.03,False,Skipped,Not Required,NaN,Corporate,115.162.101.246,2023-08-03 21:58:00
51998,TXN-20230051998,ACC-858737439,CUST-515206,Daniel Agyeman,Mortgage,active,Transfer,21/05/2025,47855.14,CHF,...,False,0.00,15.51,False,Manual Review,Pending,NaN,Corporate,166.42.248.105,2023-10-09 02:25:00


## 4. Check Summary Statistics for Numerical Columns

In [7]:
df.describe().round(2)

,amount,balance_after,fee_amount,risk_score,credit_score
count,52624.00,52624.00,52624.00,52624.00,35451.00
mean,341.08,249109.77,4.99,49.91,575.44
std,95795.83,143924.04,7.63,28.90,159.01
min,-4737986.89,107.18,0.00,0.01,300.00
25%,-25159.91,124249.14,0.00,24.82,438.00
50%,32.44,248935.86,0.00,49.94,576.00
75%,25243.50,373188.48,9.40,74.92,713.00
max,4091721.74,499987.45,25.00,100.00,850.00


## 5. Check Summary Statistics for Categorical Columns

In [8]:
df.describe(include=['str'])

,transaction_id,account_id,customer_id,customer_name,account_type,account_status,transaction_type,transaction_date,currency,transaction_channel,branch_city,merchant_name,payment_method,aml_check_status,kyc_status,customer_segment,ip_address,created_at
count,52624,52624,52624,52624,52624,52624,52624,52624,52624,52624,52624,45378,52624,52624,52624,52624,52624,52624
unique,52000,52000,50510,400,6,6,7,6208,5,7,15,9,6,6,5,5,52000,49344
top,TXN-20230001651,ACC-276905870,CUST-379091,Paul Richter,Investment,active,Payment,2024-10-21,EUR,Online,Hamburg,REWE,Debit Card,Passed,verified,Business,138.173.88.114,2023-02-27 23:19:00
freq,3,3,4,163,8896,8937,7564,46,10650,7641,3608,5135,8856,8915,10695,10620,3,4


## 6. Check Missing Values

In [9]:
df.isna().sum()

transaction_id             0
account_id                 0
customer_id                0
customer_name              0
account_type               0
account_status             0
transaction_type           0
transaction_date           0
amount                     0
currency                   0
balance_after              0
transaction_channel        0
branch_city                0
merchant_name           7246
payment_method             0
is_international           0
fee_amount                 0
risk_score                 0
fraud_flag                 0
aml_check_status           0
kyc_status                 0
credit_score           17173
customer_segment           0
ip_address                 0
created_at                 0
dtype: int64

In [10]:
(df.isna().mean()*100).round(2)

transaction_id          0.00
account_id              0.00
customer_id             0.00
customer_name           0.00
account_type            0.00
account_status          0.00
transaction_type        0.00
transaction_date        0.00
amount                  0.00
currency                0.00
balance_after           0.00
transaction_channel     0.00
branch_city             0.00
merchant_name          13.77
payment_method          0.00
is_international        0.00
fee_amount              0.00
risk_score              0.00
fraud_flag              0.00
aml_check_status        0.00
kyc_status              0.00
credit_score           32.63
customer_segment        0.00
ip_address              0.00
created_at              0.00
dtype: float64